In [2]:
# import libraries
try:
  # %tensorflow_version only exists in Colab.
  !pip install tf-nightly
except Exception:
  pass
import tensorflow as tf
import pandas as pd
from tensorflow import keras
!pip install tensorflow-datasets
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

     -------------------------------------- 352.2/352.2 MB 1.9 MB/s eta 0:00:00
     ---------------------------------------- 5.5/5.5 MB 3.6 MB/s eta 0:00:00
     ---------------------------------------- 1.5/1.5 MB 4.2 MB/s eta 0:00:00
     -------------------------------------- 226.5/226.5 kB 6.8 MB/s eta 0:00:00
     -------------------------------------- 110.8/110.8 kB 6.3 MB/s eta 0:00:00


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\saini\\AppData\\Local\\Programs\\Python\\Python310\\Lib\\site-packages\\tensorflow\\compiler\\mlir\\lite\\python\\_pywrap_converter_api.pyd'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached tensorflow_datasets-4.9.10-py3-none-any.whl (5.3 MB)
  Using cached tensorflow_metadata-1.21.0-py3-none-any.whl (30 kB)
  Using cached immutabledict-4.3.1-py3-none-any.whl (5.0 kB)
  Using cached etils-1.13.0-py3-none-any.whl (170 kB)
  Using cached dm_tree-0.1.10-cp310-cp310-win_amd64.whl (110 kB)
  Using cached promise-2.3-py3-none-any.whl
  Using cached importlib_resources-7.1.0-py3-none-any.whl (37 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl (206 kB)
  Using cached einops-0.8.2-py3-none-any.whl (65 kB)



[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


2.21.0


In [11]:
# get data files

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [13]:
# Load the training and validation data

train_data = pd.read_csv(
    train_file_path,
    sep="\t",
    header=None,
    names=["label", "message"]
)

test_data = pd.read_csv(
    test_file_path,
    sep="\t",
    header=None,
    names=["label", "message"]
)

print(train_data.shape)
print(test_data.shape)

train_data.head()

(4179, 2)
(1392, 2)


,label,message
0,ham,ahhhh...just woken up!had a bad dream about u ...
1,ham,you can never do nothing
2,ham,"now u sound like manky scouse boy steve,like! ..."
3,ham,mum say we wan to go then go... then she can s...
4,ham,never y lei... i v lazy... got wat? dat day ü ...


In [14]:
# Prepare labels

train_labels = (train_data["label"] == "spam").astype(int)
test_labels = (test_data["label"] == "spam").astype(int)

# Convert text into numerical sequences

tokenizer = keras.preprocessing.text.Tokenizer(
    num_words=10000,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(train_data["message"])

train_sequences = tokenizer.texts_to_sequences(train_data["message"])
test_sequences = tokenizer.texts_to_sequences(test_data["message"])

# Pad sequences

max_length = 100

train_padded = keras.preprocessing.sequence.pad_sequences(
    train_sequences,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

test_padded = keras.preprocessing.sequence.pad_sequences(
    test_sequences,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

# Build neural network

model = keras.Sequential([
    keras.layers.Embedding(
        input_dim=10000,
        output_dim=32,
        input_length=max_length
    ),
    keras.layers.GlobalAveragePooling1D(),
    keras.layers.Dense(24, activation="relu"),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

c:\Users\saini\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [15]:
history = model.fit(
    train_padded,
    train_labels,
    epochs=10,
    validation_data=(test_padded, test_labels),
    verbose=1
)

Epoch 1/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8588 - loss: 0.4326 - val_accuracy: 0.8657 - val_loss: 0.3658
Epoch 2/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8660 - loss: 0.3920 - val_accuracy: 0.8657 - val_loss: 0.3591
Epoch 3/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8660 - loss: 0.3792 - val_accuracy: 0.8657 - val_loss: 0.3564
Epoch 4/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8660 - loss: 0.3601 - val_accuracy: 0.8657 - val_loss: 0.3265
Epoch 5/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8660 - loss: 0.3403 - val_accuracy: 0.8657 - val_loss: 0.2954
Epoch 6/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8672 - loss: 0.2900 - val_accuracy: 0.8657 - val_loss: 0.2375
Epoch 7/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9158 - loss: 0.2155 - val_accuracy: 0.9547 - val_loss: 0.1579
Epoch 8/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9438 - loss: 0.1612 - val_accuracy: 0.

In [21]:
def predict_message(message):
    sequence = tokenizer.texts_to_sequences([message])

    padded = keras.preprocessing.sequence.pad_sequences(
        sequence,
        maxlen=max_length,
        padding="post",
        truncating="post"
    )

    probability = model.predict(padded, verbose=0)[0][0]

    if probability >= 0.1:
        label = "spam"
    else:
        label = "ham"

    return [float(probability), label]

In [22]:
print(predict_message("you have won £1000 cash! call to claim your prize."))
print(predict_message("i'll bring it tomorrow. don't forget the milk."))

[0.6272635459899902, 'spam']
[0.006173776462674141, 'ham']


In [23]:
test_messages = [
    "how are you doing today",
    "sale today! to stop texts call 98912460324",
    "i dont want to go. can we try it a different day? available sat",
    "our new mobile video service is live. just install on your phone to start watching.",
    "you have won £1000 cash! call to claim your prize.",
    "i'll bring it tomorrow. don't forget the milk.",
    "wow, is your arm alright. that happened to me one time too"
]

test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]

for msg, answer in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    print(f"Expected: {answer:4} | Predicted: {prediction[1]:4} | Probability: {prediction[0]:.4f}")
    print(msg)
    print()

Expected: ham  | Predicted: ham  | Probability: 0.0090
how are you doing today

Expected: spam | Predicted: spam | Probability: 0.1321
sale today! to stop texts call 98912460324

Expected: ham  | Predicted: ham  | Probability: 0.0105
i dont want to go. can we try it a different day? available sat

Expected: spam | Predicted: spam | Probability: 0.6015
our new mobile video service is live. just install on your phone to start watching.

Expected: spam | Predicted: spam | Probability: 0.6273
you have won £1000 cash! call to claim your prize.

Expected: ham  | Predicted: ham  | Probability: 0.0062
i'll bring it tomorrow. don't forget the milk.

Expected: ham  | Predicted: ham  | Probability: 0.0128
wow, is your arm alright. that happened to me one time too



In [24]:
for msg, answer in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    print(
        f"Expected: {answer:4} | "
        f"Predicted: {prediction[1]:4} | "
        f"Probability: {prediction[0]:.4f}"
    )

Expected: ham  | Predicted: ham  | Probability: 0.0090
Expected: spam | Predicted: spam | Probability: 0.1321
Expected: ham  | Predicted: ham  | Probability: 0.0105
Expected: spam | Predicted: spam | Probability: 0.6015
Expected: spam | Predicted: spam | Probability: 0.6273
Expected: ham  | Predicted: ham  | Probability: 0.0062
Expected: ham  | Predicted: ham  | Probability: 0.0128


In [25]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()


You passed the challenge. Great job!
